# 06. 통합 대시보드
Voilà 실행: `voila 06_dashboard.ipynb`

In [ ]:
import pandas as pd
import numpy as np
import yaml, sqlite3, os
from pathlib import Path
from dotenv import load_dotenv
from datetime import date
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly.offline import plot as offline_plot
from IPython.display import display, HTML

load_dotenv()
PROJECT_ROOT = Path.cwd()

with open(PROJECT_ROOT / 'config.yaml', encoding='utf-8') as f:
    CONFIG = yaml.safe_load(f)

MONTHLY_EXPENSE = CONFIG['user']['monthly_expense']
TARGET          = CONFIG['portfolio']
USER_NAME       = CONFIG['user']['name']
db_path         = PROJECT_ROOT / 'portfolio.db'

# ── 데이터 로드 ──────────────────────────────────────────
assets_path = PROJECT_ROOT / 'assets.csv'
sample_path = PROJECT_ROOT / 'assets_sample.csv'
df = pd.read_csv(assets_path if assets_path.exists() else sample_path, encoding='utf-8-sig')

mask = df['current_value'].isna() | (df['current_value'] == 0)
df.loc[mask, 'current_value'] = df.loc[mask, 'quantity'] * df.loc[mask, 'unit_price']
df['current_value'] = pd.to_numeric(df['current_value'], errors='coerce').fillna(0)

BUCKET_MAP = {'cash': 1, 'bond': 2, 'tdf': 2, 'equity': 3, 'income': 3}
df['bucket'] = df['asset_type'].str.lower().map(BUCKET_MAP).fillna(0).astype(int)

bucket_sum = df.groupby('bucket')['current_value'].sum()
type_sum   = df.groupby('asset_type')['current_value'].sum()
total      = df['current_value'].sum()
b1 = bucket_sum.get(1, 0)
b2 = bucket_sum.get(2, 0)
b3 = bucket_sum.get(3, 0)
months_covered = b1 / MONTHLY_EXPENSE

current_ratios = {
    'cash':   type_sum.get('cash', 0) / total,
    'bond':   (type_sum.get('bond', 0) + type_sum.get('tdf', 0)) / total,
    'equity': type_sum.get('equity', 0) / total,
    'income': type_sum.get('income', 0) / total,
}

# DB
conn = sqlite3.connect(db_path)
cur  = conn.cursor()
cur.execute('SELECT total_score, level FROM risk_scores ORDER BY date DESC LIMIT 1')
risk_row    = cur.fetchone()
total_score = risk_row[0] if risk_row else 0
risk_level  = risk_row[1] if risk_row else 'green'
cur.execute("SELECT message FROM recommendations WHERE rule_id='ai_summary' ORDER BY date DESC LIMIT 1")
ai_row     = cur.fetchone()
ai_summary = ai_row[0] if ai_row else '05_llm_summary.ipynb를 먼저 실행하세요.'
wd_df = pd.read_sql_query(
    'SELECT date, amount, guardrail_applied, note FROM withdrawal_log ORDER BY date DESC LIMIT 6', conn)
conn.close()

level_color = {'green': '#00B050', 'yellow': '#FFC000', 'red': '#FF0000'}.get(risk_level, '#00B050')
level_label = {'green': '🟢 안전', 'yellow': '🟡 주의', 'red': '🔴 위험'}.get(risk_level, '🟢 안전')
cash_emoji  = '🟢' if months_covered >= 12 else ('🟡' if months_covered >= 6 else '🔴')
today_str   = date.today().strftime('%Y년 %m월 %d일')
print('데이터 로드 완료')

In [ ]:
# ── 헤더 + 카드 ──────────────────────────────────────────
display(HTML(f"""
<div style="background:linear-gradient(135deg,#1F3864,#2E75B6);padding:24px 32px;
            border-radius:12px;margin-bottom:20px;color:white;">
  <h1 style="margin:0;font-size:24px;">🏦 은퇴 포트폴리오 대시보드</h1>
  <p style="margin:6px 0 0 0;opacity:.85;font-size:14px;">{USER_NAME}님 · {today_str}</p>
</div>
<div style="display:grid;grid-template-columns:repeat(4,1fr);gap:14px;margin-bottom:20px;">
  <div style="background:#f8f9fa;border-left:5px solid #2E75B6;border-radius:8px;padding:16px;">
    <div style="font-size:12px;color:#666;">총 자산</div>
    <div style="font-size:22px;font-weight:700;color:#1F3864;">{total/1e8:.2f}<span style="font-size:14px"> 억원</span></div>
    <div style="font-size:12px;color:#888;">월 생활비 {MONTHLY_EXPENSE/10000:.0f}만원</div>
  </div>
  <div style="background:#f8f9fa;border-left:5px solid #00B0F0;border-radius:8px;padding:16px;">
    <div style="font-size:12px;color:#666;">현금성 자산 (버킷 1)</div>
    <div style="font-size:22px;font-weight:700;color:#005F8A;">{b1/1e8:.2f}<span style="font-size:14px"> 억원</span></div>
    <div style="font-size:12px;color:#888;">{cash_emoji} {months_covered:.1f}개월치</div>
  </div>
  <div style="background:#f8f9fa;border-left:5px solid {level_color};border-radius:8px;padding:16px;">
    <div style="font-size:12px;color:#666;">위험 등급</div>
    <div style="font-size:22px;font-weight:700;color:{level_color};">{level_label}</div>
    <div style="font-size:12px;color:#888;">종합 점수 {total_score:.1f}점 / 100점</div>
  </div>
  <div style="background:#f8f9fa;border-left:5px solid #70AD47;border-radius:8px;padding:16px;">
    <div style="font-size:12px;color:#666;">성장 자산 (버킷 3)</div>
    <div style="font-size:22px;font-weight:700;color:#375623;">{b3/1e8:.2f}<span style="font-size:14px"> 억원</span></div>
    <div style="font-size:12px;color:#888;">{b3/total*100:.1f}% 비중</div>
  </div>
</div>
"""))

In [ ]:
# ── 차트 1: 파이 + 막대 + 게이지 (plotly.js 내장) ───────
labels_map = ['현금성', '채권/TDF', '주식형', '리츠/인컴']
cur_vals   = [current_ratios['cash'], current_ratios['bond'],
              current_ratios['equity'], current_ratios['income']]
tgt_vals   = [TARGET['target_cash'], TARGET['target_bond'],
              TARGET['target_equity'], TARGET['target_income']]
needle_color = {'green':'#00B050','yellow':'#FFC000','red':'#FF0000'}.get(risk_level,'#00B050')

fig1 = make_subplots(
    rows=1, cols=3,
    specs=[[{'type':'pie'}, {'type':'bar'}, {'type':'indicator'}]],
    subplot_titles=['현재 자산 배분', '목표 vs 현재', '위험 게이지']
)
fig1.add_trace(go.Pie(
    labels=labels_map, values=[v*total for v in cur_vals],
    hole=0.45, textinfo='label+percent', textfont_size=11,
    marker_colors=['#00B0F0','#70AD47','#FF6B35','#FFC000']
), row=1, col=1)
fig1.add_trace(go.Bar(name='목표', x=labels_map, y=[v*100 for v in tgt_vals],
    marker_color='#1F3864', text=[f'{v*100:.0f}%' for v in tgt_vals], textposition='outside'),
    row=1, col=2)
fig1.add_trace(go.Bar(name='현재', x=labels_map, y=[v*100 for v in cur_vals],
    marker_color='#2E75B6', text=[f'{v*100:.1f}%' for v in cur_vals], textposition='outside'),
    row=1, col=2)
fig1.add_trace(go.Indicator(
    mode='gauge+number', value=total_score,
    title={'text': level_label, 'font': {'size': 13}},
    gauge={'axis': {'range': [0,100]}, 'bar': {'color': needle_color},
           'steps': [{'range':[0,25],'color':'#E2EFDA'},
                     {'range':[25,55],'color':'#FFF2CC'},
                     {'range':[55,100],'color':'#FCE4D6'}]}
), row=1, col=3)
fig1.update_layout(height=350, font=dict(size=11), margin=dict(t=40,b=10,l=10,r=10))

# plotly.js 완전 내장 (CDN 불필요)
html1 = offline_plot(fig1, output_type='div', include_plotlyjs=True)
display(HTML(html1))

In [ ]:
# ── 차트 2: 버킷 바 차트 (plotly.js 재사용) ─────────────
fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=['버킷 1 (현금성)', '버킷 2 (채권/TDF)', '버킷 3 (성장/인컴)'],
    y=[b1/1e8, b2/1e8, b3/1e8],
    marker_color=['#00B0F0','#70AD47','#FF6B35'],
    text=[f'{v/1e8:.2f}억 ({v/MONTHLY_EXPENSE:.0f}개월)' for v in [b1,b2,b3]],
    textposition='outside', width=0.4
))
fig2.add_hline(y=12*MONTHLY_EXPENSE/1e8, line_dash='dash', line_color='red',
    annotation_text='12개월 기준', annotation_position='top right')
fig2.update_layout(title='버킷별 자산 현황', yaxis_title='금액 (억원)',
    height=300, showlegend=False, font=dict(size=12),
    margin=dict(t=50,b=10,l=10,r=10))

# include_plotlyjs=False → 위에서 이미 로드됨
html2 = offline_plot(fig2, output_type='div', include_plotlyjs=False)
display(HTML(html2))

In [ ]:
# ── AI 요약 + 인출 이력 ──────────────────────────────────
lines_html = ''.join([
    f'<p style="margin:6px 0;font-size:14px;line-height:1.7;">'
    f'<b style="color:#2E75B6;">{i+1}.</b> {line}</p>'
    for i, line in enumerate(ai_summary.split('\n')) if line.strip()
])
display(HTML(f"""
<div style="background:#EBF3FB;border-left:5px solid #2E75B6;border-radius:8px;
            padding:16px 20px;margin:16px 0;">
  <div style="font-size:12px;color:#555;margin-bottom:6px;">🤖 AI 포트폴리오 요약</div>
  {lines_html}
</div>
"""))

# 인출 이력
display(HTML('<h3 style="color:#1F3864;margin:16px 0 8px;">📋 최근 인출 이력</h3>'))
if wd_df.empty:
    display(HTML('<p style="color:#888;">인출 이력이 없습니다.</p>'))
else:
    wd_d = wd_df.copy()
    wd_d['amount'] = wd_d['amount'].apply(lambda x: f'{x:,.0f}원')
    wd_d['guardrail_applied'] = wd_d['guardrail_applied'].map({0:'정상',1:'🔴 하향'})
    wd_d.columns = ['날짜','인출금액','가드레일','비고']
    rows = ''.join(['<tr>' + ''.join([f'<td style="padding:7px 12px;font-size:13px;border-bottom:1px solid #eee;">{v}</td>' for v in r]) + '</tr>' for r in wd_d.values])
    heads = ''.join([f'<th style="padding:8px 12px;background:#1F3864;color:white;font-size:13px;text-align:left;">{c}</th>' for c in wd_d.columns])
    display(HTML(f'<table style="border-collapse:collapse;width:100%;border-radius:8px;overflow:hidden;box-shadow:0 1px 4px rgba(0,0,0,.1);"><thead><tr>{heads}</tr></thead><tbody>{rows}</tbody></table>'))

display(HTML(f'<div style="margin-top:24px;padding:12px 18px;background:#f1f3f5;border-radius:8px;font-size:12px;color:#888;">⏱ {today_str} · 📁 {"assets.csv" if (PROJECT_ROOT/"assets.csv").exists() else "샘플 데이터"} · 🗄 portfolio.db</div>'))